# SER - Experiment 03: duplicate-leakage control

**Question:** does the base paper's 94.91% come from a contaminated test split?

The RAVDESS and TESS Kaggle mirrors each ship the corpus **twice**. A recursive
scan finds 16,402 files instead of 12,162. Because the duplicates are
byte-identical recordings and the split is random over utterances, a clip and
its twin land on opposite sides of the train/test boundary about 32% of the
time - placing an estimated **41% of the test set inside the training set**.

This notebook runs the `base` configuration **deliberately un-de-duplicated**
and compares it against the clean run.

| Run | Corpus scanned | Test accuracy |
|---|---|---|
| `base` (clean, de-duplicated) | 12,162 | **0.5795** |
| `base_leaky` (this notebook) | 16,402 | ? |

If accuracy jumps toward 90%+, leakage is confirmed as the mechanism behind
the literature's inflated multi-corpus numbers.

**Attach:** the four corpora only. Do *not* attach `ser-feature-cache` - the
duplicated item list has a different fingerprint and needs fresh extraction.

**Accelerator: GPU.** Roughly 30 min extraction plus 15 min training.

In [ ]:
import glob
import json
import os
import shutil
import sys
import time

import tensorflow as tf
import keras

print("TF", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "Set Settings -> Accelerator -> GPU before running."

In [ ]:
REPO = "https://github.com/Eldorado5002/ser.git"

if not os.path.exists("/kaggle/working/ser"):
    !git clone -q {REPO} /kaggle/working/ser

sys.path.insert(0, "/kaggle/working/ser")
os.chdir("/kaggle/working/ser")
!git log --oneline -1

In [ ]:
# THE ONE DELIBERATE DIFFERENCE FROM NOTEBOOK 02.
#
# Notebook 02 symlinks the CANONICAL subdirectory of each corpus, which holds
# exactly one copy. Here we symlink its PARENT - the dataset root - so the
# recursive scan sees BOTH copies of RAVDESS and TESS. Everything else is
# identical to the `base` configuration.
DATA_ROOT = "/kaggle/working/ser/data_leaky"

CANONICAL = {
    "RAVDESS": "audio_speech_actors_01-24",
    "TESS":    "TESS Toronto emotional speech set data",
    "SAVEE":   "ALL",
    "CREMA-D": "AudioWAV",
}


def find_canonical(target):
    hits = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        for d in dirs:
            if d.lower() == target.lower():
                hits.append(os.path.join(root, d))
    return sorted(hits)[0] if hits else None


os.makedirs(DATA_ROOT, exist_ok=True)
for name, target in CANONICAL.items():
    canon = find_canonical(target)
    assert canon is not None, f"MISSING INPUT for {name}: no '{target}' found"
    src = os.path.dirname(canon)          # the PARENT, i.e. the dataset root
    dst = os.path.join(DATA_ROOT, name)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)
    os.symlink(src, dst)
    n = len(glob.glob(os.path.join(dst, "**", "*.wav"), recursive=True))
    print(f"{name:9s} {n:6d} wav   <- {src}")

In [ ]:
import config
import data_loader

# Repoint the scanner at the duplicated tree.
data_loader._CORPORA = [
    ("RAVDESS", os.path.join(DATA_ROOT, "RAVDESS"), data_loader._parse_ravdess),
    ("TESS",    os.path.join(DATA_ROOT, "TESS"),    data_loader._parse_tess),
    ("SAVEE",   os.path.join(DATA_ROOT, "SAVEE"),   data_loader._parse_savee),
    ("CREMA-D", os.path.join(DATA_ROOT, "CREMA-D"), data_loader._parse_cremad),
]

config.CACHE_DIR = "/kaggle/working/features_cache_leaky"
config.RUNS_DIR = "/kaggle/working/runs"
os.makedirs(config.CACHE_DIR, exist_ok=True)
os.makedirs(config.RUNS_DIR, exist_ok=True)

# strict=False deliberately DISABLES the duplicate guard - that is the point.
meta = data_loader.build_metadata(strict=False)
print()
print(f"scanned {len(meta)} samples  (the clean run scanned 12162)")
assert len(meta) > 12162, (
    "Expected MORE than 12,162 samples. The symlinks are pointing at the "
    "canonical subdirectories rather than the dataset roots, so there is "
    "nothing to test.")

names = meta.path.map(os.path.basename)
print(f"duplicated basenames: {(names.value_counts() > 1).sum()}")

In [ ]:
from data_loader import split_metadata

train_df, val_df, test_df = split_metadata(meta)

# Quantify the contamination directly: how many TEST recordings have an
# identical twin (same basename) somewhere in train or val?
seen = set(train_df.path.map(os.path.basename)) | \
       set(val_df.path.map(os.path.basename))
leaked = test_df.path.map(os.path.basename).isin(seen).sum()

print()
print(f"test samples             : {len(test_df)}")
print(f"with a twin in train/val : {leaked}")
print(f"CONTAMINATION            : {100 * leaked / len(test_df):.1f}% "
      f"of the test set")

In [ ]:
from train import run_experiment

# Identical to the `base` configuration: no novelties. The ONLY difference
# from notebook 02's `base` run is the duplicated corpus.
metrics = run_experiment(
    use_afw=False, use_eaaa=False, use_mstc=False, use_cadl=False,
    tag="base_leaky", epochs=config.EPOCHS, metadata=meta)

In [ ]:
CLEAN_BASE = 0.579531   # notebook 02, de-duplicated 12,162-sample corpus
PAPER = 0.9491          # Chourasia et al. (2026)

leaky = metrics["accuracy"]
delta = leaky - CLEAN_BASE

print("=" * 66)
print("  DUPLICATE-LEAKAGE CONTROL EXPERIMENT")
print("=" * 66)
print(f"  base       (clean,  12,162 samples) : {CLEAN_BASE:.4f}")
print(f"  base_leaky (dupes,  {len(meta):,} samples) : {leaky:.4f}")
print(f"  difference                          : {delta:+.4f}"
      f"   ({100 * delta:+.2f} points)")
print(f"  base paper reported                 : {PAPER:.4f}")
print("=" * 66)
print()

if delta > 0.15:
    print("  CONFIRMED: duplicate contamination inflates accuracy massively.")
    print("  The literature's high multi-corpus numbers are consistent with")
    print("  scanning an un-de-duplicated mirror.")
elif delta > 0.05:
    print("  PARTIAL: contamination inflates accuracy, but does not on its")
    print("  own account for the gap to 94.91%.")
else:
    print("  NOT CONFIRMED: duplication is not the main driver. The gap to")
    print("  94.91% needs another explanation.")

json.dump({"clean_base": CLEAN_BASE, "leaky_base": float(leaky),
           "paper": PAPER, "n_samples_leaky": int(len(meta)),
           "delta": float(delta)},
          open("/kaggle/working/leakage_experiment.json", "w"), indent=2)

In [ ]:
# Drop the symlinks so /kaggle/working does not follow them when saving.
for name in CANONICAL:
    link = os.path.join(DATA_ROOT, name)
    if os.path.islink(link):
        os.unlink(link)

print("output:", sorted(os.listdir("/kaggle/working")))